In [1]:
with open(r"C:\Users\Mykola\Downloads\FOP.xml", 'r', encoding='utf-8') as f:
    for _ in range(15):  # Виведемо перші 15 рядків
        print(f.readline().strip())

<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<DATA>
<SUBJECT>
<ESTATE_MANAGER></ESTATE_MANAGER>
<EXCHANGE_DATA/>
<FARMER></FARMER>
<NAME>ЖМУЦЬКА НАТАЛІЯ ВАСИЛІВНА</NAME>
<RECORD>14426646</RECORD>
<REGISTRATION>13.09.2024; 13.09.2024; 2010350000000641237</REGISTRATION>
<STAN>зареєстровано</STAN>
<TERMINATED_INFO></TERMINATED_INFO>
<TERMINATION_CANCEL_INFO></TERMINATION_CANCEL_INFO>
</SUBJECT>

<SUBJECT>


In [14]:
import pandas as pd
import xml.etree.ElementTree as ET
import os

def xml_to_csv_chunked(xml_path, csv_path, record_tag='SUBJECT', chunk_size=50000):
    """
    Парсить великий XML файл та зберігає його частинами у CSV.
    """
    print(f"Починаємо обробку файлу: {xml_path}")
    
    context = ET.iterparse(xml_path, events=('end',))
    
    batch = []
    chunk_counter = 1
    total_records = 0
    write_header = not os.path.exists(csv_path) 
    
    for event, elem in context:
        # Шукаємо завершення тегу <SUBJECT>
        if elem.tag == record_tag:
            row_data = {}
            
            # Збираємо всі "листочки" (NAME, STAN, RECORD тощо)
            for child in elem:
                # Якщо тег порожній (наприклад <EXCHANGE_DATA/>), запишемо None
                row_data[child.tag] = child.text.strip() if child.text else None
                
            batch.append(row_data)
            total_records += 1
            
            # Звільняємо пам'ять! (відрізаємо оброблену гілку)
            elem.clear()
            
            # Якщо назбирали достатньо даних — скидаємо на диск
            if len(batch) >= chunk_size:
                df = pd.DataFrame(batch)
                df.to_csv(csv_path, mode='a', index=False, header=write_header, encoding='utf-8')
                
                print(f"Збережено чанк {chunk_counter}: +{chunk_size} записів (Загалом: {total_records})")
                
                batch = [] 
                write_header = False 
                chunk_counter += 1
                
    # Зберігаємо "хвостик" даних, який залишився після останнього повного батчу
    if batch:
        df = pd.DataFrame(batch)
        df.to_csv(csv_path, mode='a', index=False, header=write_header, encoding='utf-8')
        print(f"Збережено фінальний чанк {chunk_counter}: +{len(batch)} записів (Загалом: {total_records})")
        
    print(f"\nОбробку завершено! Успішно перетворено {total_records} записів.")

# === Запуск ===
input_xml = r"C:\Users\Mykola\Downloads\UO.xml"
output_csv = "UO_parsed.csv"

# Викликаємо функцію з правильним тегом SUBJECT
xml_to_csv_chunked(input_xml, output_csv, record_tag='SUBJECT', chunk_size=50000)

Починаємо обробку файлу: C:\Users\Mykola\Downloads\UO.xml
Збережено чанк 1: +50000 записів (Загалом: 50000)
Збережено чанк 2: +50000 записів (Загалом: 100000)
Збережено чанк 3: +50000 записів (Загалом: 150000)
Збережено чанк 4: +50000 записів (Загалом: 200000)


KeyboardInterrupt: 

In [3]:
df = pd.read_csv(r"C:\OTP\FOP_parsed.csv")
df

C:\Users\Mykola\AppData\Local\Temp\ipykernel_39160\578389982.py:1: DtypeWarning: Columns (2,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"C:\OTP\FOP_parsed.csv")


,ESTATE_MANAGER,EXCHANGE_DATA,FARMER,NAME,RECORD,REGISTRATION,STAN,TERMINATED_INFO,TERMINATION_CANCEL_INFO
0,NaN,NaN,NaN,ЖМУЦЬКА НАТАЛІЯ ВАСИЛІВНА,14426646,13.09.2024; 13.09.2024; 2010350000000641237,зареєстровано,NaN,NaN
1,NaN,NaN,NaN,ДМИТРИШИН АНДРІЙ БОГДАНОВИЧ,12633820,02.04.2019; 02.04.2019; 24010000000004846,припинено,13.09.2024; 2004010060002004846; Державна реєс...,NaN
2,NaN,NaN,NaN,Кирияк Анастасія Олексіївна,14426647,13.09.2024; 13.09.2024; 2010350000000641238,зареєстровано,NaN,NaN
3,NaN,NaN,NaN,САВЧУК ВОЛОДИМИР ПЕТРОВИЧ,13066292,28.07.2020; 28.07.2020; 21900000000003692,припинено,13.09.2024; 2001900060001003692; Державна реєс...,NaN
4,NaN,NaN,NaN,СУРЖКО ОЛЕКСАНДР АНАТОЛІЙОВИЧ,13063074,24.07.2020; 24.07.2020; 22240000000145439,припинено,05.02.2026; 2002240060002145439; Державна реєс...,NaN
...,...,...,...,...,...,...,...,...,...
2949995,NaN,NaN,NaN,РАРЕНКО ОКСАНА ВІТАЛІЇВНА,6570536,22.07.2010; 22.07.2010; 26760000000010914,припинено,08.04.2011; 26760060002010914; Державна реєстр...,NaN
2949996,NaN,NaN,NaN,ЩЕПАНОВСЬКА ІРИНА ВАСИЛІВНА,6501357,14.07.2010; 14.07.2010; 23450000000002136,припинено,23.06.2017; 23450060004002136; Державна реєстр...,NaN
2949997,NaN,NaN,NaN,ТАРАН ІГОР ОЛЕКСАНДРОВИЧ,6598498,26.07.2010; 26.07.2010; 20910000000002648,зареєстровано,NaN,NaN
2949998,NaN,NaN,NaN,УЩЕНКО МАКСИМ АНАТОЛІЙОВИЧ,6600237,26.07.2010; 26.07.2010; 22280000000003069,припинено,08.12.2011; 22280060003003069; Державна реєстр...,NaN


In [15]:
uo = pd.read_csv(r"C:\OTP\UO_parsed.csv")
uo

C:\Users\Mykola\AppData\Local\Temp\ipykernel_39160\3185455573.py:1: DtypeWarning: Columns (10,25) have mixed types. Specify dtype option on import or set low_memory=False.
  uo = pd.read_csv(r"C:\OTP\UO_parsed.csv")


,ASSIGNEES,AUTHORIZED_CAPITAL,BANKRUPTCY_READJUSTMENT_INFO,BENEFICIARIES,BRANCHES,EDRPOU,EXCHANGE_DATA,EXECUTIVE_POWER,FOUNDERS,FOUNDING_DOCUMENT_NUM,...,REGISTRATION,SHORT_NAME,SIGNERS,STAN,STATUTE,SUPERIOR_MANAGEMENT,TERMINATED_INFO,TERMINATION_CANCEL_INFO,TERMINATION_STARTED_INFO,PURPOSE
0,NaN,"5420500,00",NaN,NaN,NaN,45504376.0,NaN,NaN,NaN,0100000000000000000000099900000001111199000000...,...,13.09.2024; 13.09.2024; 1010351020000011182,"ТОВ ""СГК ЕКІПАЖ""",NaN,зареєстровано,Діє на підставі модельного статуту,загальні збори учасників; директор,NaN,NaN,NaN,NaN
1,NaN,"305000,00",NaN,NaN,NaN,42769785.0,NaN,NaN,NaN,NaN,...,23.01.2019; 23.01.2019; 10701020000080768,"ТОВ ""ВІКОМС""",NaN,зареєстровано,"Діє на підставі установчих документів, затверд...","ЗАГАЛЬНІ ЗБОРИ УЧАСНИКІВ, ДИРЕКТОР",NaN,NaN,NaN,NaN
2,NaN,"888,00",NaN,NaN,NaN,37235839.0,NaN,NaN,NaN,NaN,...,10.08.2010; 10.08.2010; 13531020000003938,"ТОВ ""БК ""ГЕРМЕС-БУД""",NaN,зареєстровано,"Діє на підставі установчих документів, затверд...",NaN,NaN,NaN,NaN,NaN
3,NaN,"275800,00",NaN,NaN,NaN,44137871.0,NaN,NaN,NaN,0100000000000000000000099900000001111199000000...,...,16.11.2020; 16.11.2020; 1003551020000011075,"ТОВ ""КОМФОРТ ТАУН ОПТІ""",NaN,зареєстровано,Діє на підставі модельного статуту,загальні збори учасників; директор,NaN,NaN,NaN,NaN
4,NaN,"295600,00",NaN,NaN,NaN,43914641.0,NaN,NaN,NaN,0100000000000000000000099900000001111199000000...,...,17.11.2020; 17.11.2020; 1003551020000011079,"ТОВ ""ПАРТНЕР ІНВЕСТ ПРЕМІУМ""",NaN,зареєстровано,Діє на підставі модельного статуту,загальні збори учасників; директор,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,NaN,"1000,00",NaN,NaN,NaN,35358725.0,NaN,NaN,NaN,NaN,...,14.09.2007; 14.09.2007; 15561020000029212,"ПП ""К.К.К.""",NaN,припинено,"Діє на підставі установчих документів, затверд...",NaN,23.06.2008; 15561170003029212; Державна реєстр...,NaN,NaN,NaN
199996,NaN,"0,00",NaN,NaN,NaN,22080436.0,NaN,NaN,NaN,NaN,...,11.09.1995; 14.09.2007; 13251200000000488,"ПП ФІРМА ""КОМТЕХ""",NaN,припинено,"Діє на підставі установчих документів, затверд...",NaN,05.10.2011; 13251720004000488; Державна реєстр...,NaN,NaN,NaN
199997,NaN,"44000,00",NaN,NaN,NaN,35267324.0,NaN,NaN,NaN,NaN,...,05.07.2007; 05.07.2007; 12241020000037762,"ТОВ ""СТАНКОГІДРОСЕРВІС - ЄС""",NaN,зареєстровано,"Діє на підставі установчих документів, затверд...",NaN,NaN,NaN,NaN,NaN
199998,NaN,"50000,00",NaN,NaN,NaN,35267345.0,NaN,NaN,NaN,NaN,...,05.07.2007; 05.07.2007; 12241020000037763,"ПП ""ТОРГОВИЙ ДІМ МІТАС""",NaN,припинено,"Діє на підставі установчих документів, затверд...",NaN,20.06.2017; 12241720004037763; Державна реєстр...,NaN,NaN,NaN


In [ ]:
uo[uo["BENEFICIARIES"]]

In [13]:
df[df["RECORD"].astype(str).str.len() == 10]

,ESTATE_MANAGER,EXCHANGE_DATA,FARMER,NAME,RECORD,REGISTRATION,STAN,TERMINATED_INFO,TERMINATION_CANCEL_INFO


In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd

xml_file = r"C:\Users\Mykola\Downloads\FOP.xml"
output_csv = r"fop_data.csv"

def clean_tag(tag):
    return tag.split("}", 1)[-1]

rows = []
counter = 0

context = ET.iterparse(xml_file, events=("end",))

for event, elem in context:
    tag = clean_tag(elem.tag)

    if tag in ["RECORD", "ROW", "ENTRY", "FOP"]:
        row_data = {}

        for child in elem:
            child_tag = clean_tag(child.tag)
            row_data[child_tag] = (child.text or "").strip()

        if row_data:
            rows.append(row_data)
            counter += 1

        # 🔹 кожні 1000 записів — показуємо прогрес
        if counter % 10 == 0 and counter > 0:
            print(f"Processed: {counter} rows")

            # показати приклад
            print("Sample row:", rows[-1])

        # 🔹 кожні 50k — пишемо у файл (щоб не втратити прогрес)
        if counter % 50 == 0 and counter > 0:
            df_chunk = pd.DataFrame(rows)
            df_chunk.to_csv(output_csv, mode="a", index=False, header=(counter == 50000), encoding="utf-8-sig")
            print(f"Saved chunk up to {counter}")

            rows = []  # очищаємо пам'ять

        elem.clear()

# 🔹 дозапис залишків
if rows:
    df = pd.DataFrame(rows)
    df.to_csv(output_csv, mode="a", index=False, header=False, encoding="utf-8-sig")

print(f"Done. Total rows: {counter}")